# Fields 统计与可视化（示例）

本 notebook 用于演示如何读取 PhiFlow 风格的 HDF5 输出（以 `fields.h5` 为例），并对速度分量（Ux, Uy）与压力（p）做简单统计与可视化。（示例中仅展示最后一个时间帧的可视化）

本 notebook 包含：
- 读取 HDF5 中的 velocity、pressure 数据；
- 计算速度幅值（speed）和二维涡量（vorticity z 分量）；
- 计算并展示统计指标（max/min/mean/median/top10/1%计数）；
- 使用 Matplotlib 绘制热力图并保存到 `analysis/` 文件夹。

在本 notebook 所在的 `examples/` 目录中，可用以下命令执行全部单元：

```bash
jupyter nbconvert --to notebook --execute inspect_runner_fields.ipynb --output executed_inspect_runner_fields.ipynb
```

## 1) 配置: 路径 与 可视化 输出目录

将下面的 `H5_PATH` 指向你运行 `SolverRunner` 后生成的 `fields.h5`。 默认路径对应从 `examples/` 运行方柱算例时生成的文件。

In [ ]:
import h5py, json
from pathlib import Path
from pprint import pprint
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (8,4)
plt.rcParams['figure.dpi'] = 110

H5_PATH = Path('outputs/karman_run_drl_square/raw/fields.h5')
if not H5_PATH.is_file():
    raise FileNotFoundError(f'请将 H5_PATH 指向已有的流场文件：{H5_PATH}')
OUT_DIR = H5_PATH.parent.parent / 'analysis'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('HDF5 exists:', H5_PATH.exists())
print('Using OUT_DIR =', OUT_DIR)

## 2) 打开文件并显示结构（通用处理）

In [ ]:
with h5py.File(str(H5_PATH), 'r') as f:
    print('Root keys:', list(f.keys()))

    # 支持两种布局：
    # A: time dimension datasets under root: e.g. velocity_center (t, comps, ny, nx), pressure (t, ny, nx)
    # B: step groups: step_1, step_2, ..., each contains 'velocity' & 'pressure' datasets.
    has_groups = any(k.startswith('step_') for k in f.keys())
    if has_groups:
        # 按数值排序时间步，避免字典序的问题（例如 'step_90' 被排在 'step_180' 之前）
        steps = sorted([int(k.split('_')[1]) for k in f.keys() if k.startswith('step_')])
        steps = [f'step_{s}' for s in steps]
        print('HDF5 contains step groups, found %d steps' % len(steps))
        last_group = steps[-1]
        print('Last group:', last_group)
        g = f[last_group]
        print('Group keys:', list(g.keys()))
    else:
        # root layout: check 'velocity_center' or 'velocity' & 'pressure' directly
        print('Root layout assumed. Keys: ', list(f.keys()))

# 如果需要可以打印更多关于 shapes 的信息（在下面读取数据时显示）。

## 3) 读取最后一帧数据（兼容两种布局）

In [ ]:
def read_last_frame(h5_path: Path):
    with h5py.File(str(h5_path), 'r') as f:
        if any(k.startswith('step_') for k in f.keys()):
            steps = sorted((k for k in f if k.startswith('step_')), key=lambda k: int(k[5:]))
            last = steps[-1]
            grp = f[last]
            # 常见 dataset 名称： 'velocity', 'pressure' （velocity shape: 2, ny, nx）
            v = np.array(grp['velocity'][()]) if 'velocity' in grp else None
            p = np.array(grp['pressure'][()]) if 'pressure' in grp else None
            return v, p, dict(f.attrs)
        else:
            # root datasets：可能是 velocity_center（t, comps, ny, nx）和 pressure（t, ny, nx）
            v = np.array(f['velocity_center'][()]) if 'velocity_center' in f else (np.array(f['velocity'][()]) if 'velocity' in f else None)
            p = np.array(f['pressure'][()]) if 'pressure' in f else None
            return v, p, dict(f.attrs)

v, p, attrs = read_last_frame(H5_PATH)
print('Velocity shape:', None if v is None else v.shape)
print('Pressure shape:', None if p is None else p.shape)
print('Grid:', {k: attrs[k] for k in ('resolution', 'domain_size') if k in attrs})

## 4) 计算统计量与简单可视化（最后时间帧）

In [ ]:
# 兼容并提取 vx, vy, pressure (确保输出为 (ny, nx) 形式)
if v is None or p is None:
    raise RuntimeError('HDF5 中找不到 velocity 或 pressure 数据集，请确认。')

# 读取 SDF / mask (优先 group 中的 dataset)
sdf = None
with h5py.File(str(H5_PATH), 'r') as f:
    if any(k.startswith('step_') for k in f.keys()):
        steps = sorted((k for k in f if k.startswith('step_')), key=lambda k: int(k[5:]))
        grp = f[steps[-1]]
        if 'sdf' in grp:
            sdf = np.array(grp['sdf'][()])
        elif 'phi' in grp:
            sdf = np.array(grp['phi'][()])
        elif 'mask' in grp:
            sdf = np.array(grp['mask'][()])
    else:
        if 'sdf' in f:
            sdf = np.array(f['sdf'][()])
        elif 'phi' in f:
            sdf = np.array(f['phi'][()])
        elif 'mask' in f:
            sdf = np.array(f['mask'][()])

# 如果读取到 sdf（signed distance），fluid mask 为 sdf > 0；
# 如果读取到 mask（0/1），则 mask > 0 视为流体
fluid_mask = None
if sdf is not None:
    # 如果 sdf 为 (t, ny, nx) 或 (ny, nx) 兼容处理
    if sdf.ndim == 3 and sdf.shape[0] > 1:
        sdf_arr = sdf[-1, :, :]
    else:
        sdf_arr = sdf
    fluid_mask = sdf_arr > 0


def extract_vx_vy(v_arr):
    # 支持多种存储形式，返回维度 (ny, nx) 的 vx, vy
    if v_arr.ndim == 4:
        # (t, comps, ny, nx)
        ft = v_arr.shape[0] - 1
        vx = v_arr[ft, 0, :, :]
        vy = v_arr[ft, 1, :, :]
    elif v_arr.ndim == 3 and v_arr.shape[0] in (2,3):
        # (2, ny, nx)
        vx = v_arr[0, :, :]
        vy = v_arr[1, :, :]
    elif v_arr.ndim == 3 and v_arr.shape[-1] == 2:
        # (ny, nx, 2)
        vx = v_arr[..., 0]
        vy = v_arr[..., 1]
    elif v_arr.ndim == 2 and v_arr.shape[-1] == 2:
        vx = v_arr[..., 0]
        vy = v_arr[..., 1]
    else:
        raise RuntimeError('Unsupported velocity shape: %s' % (str(v_arr.shape)))
    return vx, vy

vx, vy = extract_vx_vy(v)
# pressure 最终为 (ny, nx)
if p.ndim == 3:
    pr = p[-1, :, :]
else:
    pr = p

# 标准化形状：基于 attrs['resolution'] 和数组 shape 确保 (ny, nx)
res = attrs.get('resolution', None)
if res is not None:
    try:
        rx, ry = int(res[0]), int(res[1])
        # res stored as [Nx, Ny]
        nx_attr, ny_attr = rx, ry
        # 如果 pr 的 shape 与 (Nx, Ny) 对应，则做转置为 (Ny, Nx)
        if pr.shape == (nx_attr, ny_attr):
            pr = pr.T
        if vx.shape == (nx_attr, ny_attr):
            vx = vx.T
            vy = vy.T
        # sdf 对齐
        if fluid_mask is not None and (fluid_mask.shape == (nx_attr, ny_attr)):
            fluid_mask = fluid_mask.T
    except Exception:
        pass

# 现在 pr / vx / vy 的 shape 应为 (ny, nx)
ny, nx = pr.shape

# 构建物理坐标（优先使用 attrs['domain_size']），cell centers: (i+0.5)*dx
domain_size = attrs.get('domain_size', None)
if domain_size is not None and len(domain_size) >= 2:
    Lx, Ly = float(domain_size[0]), float(domain_size[1])
    dx = Lx / nx
    dy = Ly / ny
    x_coords = dx * (0.5 + np.arange(nx))
    y_coords = dy * (0.5 + np.arange(ny))
else:
    x_coords = np.arange(nx).astype(float)
    y_coords = np.arange(ny).astype(float)

speed = np.sqrt(vx**2 + vy**2)

# 计算 vorticity z (approx): dv/dx - du/dy 注意 axis= (ny,nx) -> axis0=y, axis1=x

dvdx = np.gradient(vy, x_coords, axis=1)
dudy = np.gradient(vx, y_coords, axis=0)
vorticity = dvdx - dudy

# 统计帮助函数：支持 mask

def get_topn_with_coords(arr, x_coords, y_coords, n=10, mask=None):
    a = np.nan_to_num(arr)
    if mask is not None:
        if mask.shape != a.shape:
            # try flip
            mask = mask.T if mask.shape[::-1] == a.shape else None
    flat = a.flatten()
    mask_flat = (mask.flatten() if mask is not None else np.ones_like(flat, dtype=bool))
    valid_idxs = np.where(mask_flat)[0]
    if valid_idxs.size == 0:
        return []
    if n >= valid_idxs.size:
        idxs = valid_idxs[np.argsort(flat[valid_idxs])[::-1]]
    else:
        part = np.argpartition(flat[valid_idxs], -n)[-n:]
        idxs = valid_idxs[part]
        idxs = idxs[np.argsort(flat[idxs])[::-1]]
    res = []
    for idx in idxs:
        val = float(flat[idx])
        iy, ix = np.unravel_index(idx, a.shape)
        cx, cy = float(x_coords[ix]), float(y_coords[iy])
        res.append({'val': val, 'ix': int(ix), 'iy': int(iy), 'coord': (cx, cy)})
    return res


def stats_basic(arr, mask=None):
    a = np.array(arr, dtype=float)
    if mask is not None:
        if mask.shape != a.shape:
            # try transposing mask if mismatch
            if mask.shape[::-1] == a.shape:
                mask = mask.T
            else:
                raise RuntimeError('Mask shape mismatch: %s vs %s' % (str(mask.shape), str(a.shape)))
        flat = a[mask].flatten()
    else:
        flat = a.flatten()
    if flat.size == 0:
        return {'min': None, 'max': None, 'mean': None, 'median': None, 'count': 0}
    return {'min': float(np.nanmin(flat)), 'max': float(np.nanmax(flat)), 'mean': float(np.nanmean(flat)), 'median': float(np.nanmedian(flat)), 'count': int(flat.size)}

# generate summary with vx, vy stats and masked fluid-only stats
summary = {}
summary['vx'] = stats_basic(vx, mask=fluid_mask)
summary['vy'] = stats_basic(vy, mask=fluid_mask)
summary['speed'] = stats_basic(speed, mask=fluid_mask)
summary['pressure'] = stats_basic(pr, mask=fluid_mask)
summary['vorticity'] = stats_basic(vorticity, mask=fluid_mask)

# top10 与 1% 阈值计数（对 speed 使用）
summary['top10_speed'] = get_topn_with_coords(speed, x_coords, y_coords, n=10, mask=fluid_mask)
summary['top10_vx'] = get_topn_with_coords(vx, x_coords, y_coords, n=10, mask=fluid_mask)
summary['top10_vy'] = get_topn_with_coords(vy, x_coords, y_coords, n=10, mask=fluid_mask)

# 对于 1% 阈值统计，我们只检查 fluid cells
valid_speed = speed[fluid_mask] if fluid_mask is not None else speed.flatten()
thr_speed = float(np.nanpercentile(valid_speed, 99)) if valid_speed.size>0 else 0.0
summary['1pct_thresholds'] = {'speed': thr_speed}
summary['1pct_counts'] = {'speed': int(np.sum(valid_speed >= thr_speed)) if valid_speed.size>0 else 0}

# 预览结果
from pprint import pprint

print('vx stats:')
pprint(summary['vx'])

print('vy stats:')
pprint(summary['vy'])

print('speed stats:')
pprint(summary['speed'])

print('pressure stats:')
pprint(summary['pressure'])

print('1% speed thr/count:')
pprint((thr_speed, summary['1pct_counts']['speed']))


In [ ]:
# 写入 summary JSON，便于后续分析脚本读取
json_path = OUT_DIR / 'fields_summary_extracted.json'
# add mask summary
if fluid_mask is not None:
    mask_count = int(np.sum(fluid_mask))
    mask_total = int(fluid_mask.size)
    summary['mask_summary'] = {'fluid_cells': mask_count, 'total_cells': mask_total, 'fraction': float(mask_count/mask_total)}
else:
    summary['mask_summary'] = None

with open(json_path, 'w', encoding='utf-8') as fh:
    json.dump(summary, fh, ensure_ascii=False, indent=2)
print('Summary saved to:', json_path)

# 保存 mask 可视化
if fluid_mask is not None:
    fig, ax = plt.subplots(figsize=(5,5))
    dx = x_coords[1] - x_coords[0] if len(x_coords)>1 else 1.0
    dy = y_coords[1] - y_coords[0] if len(y_coords)>1 else 1.0
    extent = (x_coords[0] - dx/2, x_coords[-1] + dx/2, y_coords[0] - dy/2, y_coords[-1] + dy/2)
    ax.imshow(fluid_mask.astype(int), origin='lower', extent=extent, cmap='Greys', interpolation='nearest')
    ax.set_title('Fluid mask (1=fluid)')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    fig.savefig(OUT_DIR / 'fluid_mask.png', bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved fluid mask map to', OUT_DIR / 'fluid_mask.png')


In [ ]:
# 绘图函数：使用 extent 保持物理坐标轴一致（确保 arr 是 (ny, nx)）
def plot_field(arr, x_coords, y_coords, title, cmap='viridis', save_name=None, vmin=None, vmax=None, mask=None):
    fig, ax = plt.subplots(figsize=(6,4))
    dx = x_coords[1] - x_coords[0] if len(x_coords) > 1 else 1.0
    dy = y_coords[1] - y_coords[0] if len(y_coords) > 1 else 1.0
    extent = (x_coords[0] - dx/2, x_coords[-1] + dx/2,
              y_coords[0] - dy/2, y_coords[-1] + dy/2)
    im = ax.imshow(arr, origin='lower', extent=extent, aspect='equal', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    fig.colorbar(im, ax=ax)
    ax.set_title(title)
    # overlay mask as semi-transparent gray where solid (mask is True for fluid)
    if mask is not None:
        # mask True means fluid; we want to show solid where mask is False
        solid = (~mask).astype(float)
        # create an extent-mapped array; imshow expects (ny, nx)
        ax.imshow(solid, origin='lower', extent=extent, aspect='equal', cmap='Greys', alpha=0.45)
    if save_name is not None:
        fig.savefig(OUT_DIR / save_name, bbox_inches='tight')
    plt.show()
    plt.close(fig)

# 依据 attrs['resolution'] 判断数组方向并在必要时转置（runner 存储时 resolution=[Nx,Ny]）。
res = attrs.get('resolution', attrs.get('grid_resolution', None))
if res is not None:
    try:
        rx, ry = int(res[0]), int(res[1])
        nx_attr, ny_attr = rx, ry
        # 如果 pressure 的 shape 恰好是 (Nx, Ny)，则做转置以得到 (Ny, Nx)
        if pr.shape == (nx_attr, ny_attr):
            vx = vx.T
            vy = vy.T
            pr = pr.T
            speed = speed.T
            vorticity = vorticity.T
            if fluid_mask is not None and fluid_mask.shape == (nx_attr, ny_attr):
                fluid_mask = fluid_mask.T
    except Exception:
        pass

# 重新确保 shapes
ny, nx = pr.shape

# 计算 vorticity（再次确保 dx,dy 以物理坐标为准）
# using central difference based on coordinates

dx = x_coords[1] - x_coords[0] if len(x_coords) > 1 else 1.0
dy = y_coords[1] - y_coords[0] if len(y_coords) > 1 else 1.0

dvdx = np.gradient(vy, dx, axis=1)
dudy = np.gradient(vx, dy, axis=0)
vorticity = dvdx - dudy

# 绘制 core fields：speed, vx, vy, vorticity, pressure
plot_field(speed, x_coords, y_coords, 'Speed (last frame)', cmap='plasma', save_name='speed_last.png', mask=fluid_mask)
plot_field(vx, x_coords, y_coords, 'Vx (last frame)', cmap='coolwarm', save_name='vx_last.png', mask=fluid_mask)
plot_field(vy, x_coords, y_coords, 'Vy (last frame)', cmap='coolwarm', save_name='vy_last.png', mask=fluid_mask)
plot_field(vorticity, x_coords, y_coords, 'Vorticity z (last frame)', cmap='RdBu_r', save_name='vorticity_last.png', mask=fluid_mask)
plot_field(pr, x_coords, y_coords, 'Pressure (last frame)', cmap='viridis', save_name='pressure_last.png', mask=fluid_mask)

# 绘制流线图（streamlines），颜色按速度着色并保存
# 确保 vx, vy 的 shape 为 (ny, nx)，x_coords 长度为 nx，y_coords 长度为 ny
try:
    X, Y = np.meshgrid(x_coords, y_coords)
    # 归一化线宽（避免全部为零时报错）
    local_speed = np.sqrt(vx**2 + vy**2)
    max_speed = np.nanmax(local_speed) if np.nanmax(local_speed) > 0 else 1.0
    lw = 1.5 * (local_speed / max_speed)
    fig, ax = plt.subplots(figsize=(6,4))
    strm = ax.streamplot(X, Y, vx, vy, color=local_speed, linewidth=lw, cmap='plasma', density=1.25, arrowstyle='-|>', arrowsize=1)
    # overlay solid region
    if fluid_mask is not None:
        solid = (~fluid_mask).astype(float)
        extent = (x_coords[0] - dx/2, x_coords[-1] + dx/2, y_coords[0] - dy/2, y_coords[-1] + dy/2)
        ax.imshow(solid, origin='lower', extent=extent, aspect='equal', cmap='Greys', alpha=0.45)
    fig.colorbar(strm.lines, ax=ax, label='speed')
    ax.set_title('Streamlines (last frame) - color by speed')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    fig.savefig(OUT_DIR / 'streamlines_last.png', bbox_inches='tight')
    plt.show()
    plt.close(fig)
    # print('Saved streamlines to', OUT_DIR / 'streamlines_last.png')
except Exception as e:
    print('Failed to draw streamlines:', e)

print('Saved plots to', OUT_DIR)